# The Brain of AI agents: LLM

Here, we
- Select the right LLM for agent development
- Use LiteLLM to model against a common LLM API facade
- Engineer prompts for agents
- Setup GAIA benchmark and make LLMs limitations obvious

In [1]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(
  model="qwen3",
  messages=[
    {"role": "system", "content": "You are a helpful assistant, providing concise answers."},
    {"role": "user", "content": "Why is the sky blue?"}
  ],
    stream=True,
)

for chunk in response:
  print(chunk.message.content, end="", flush=True)

The sky appears blue due to **Rayleigh scattering**, where sunlight interacts with molecules and small particles in Earth's atmosphere. Shorter wavelengths (like blue/violet) scatter more than longer wavelengths (red/orange). While violet light scatters the most, our eyes are more sensitive to blue, and some violet light is absorbed by the atmosphere. Combined, this makes the sky appear blue during the day. At sunrise/sunset, longer paths through the atmosphere scatter blue light out of our line of sight, leaving red/orange hues.

## Nutze LiteLLM

Wir nutzen nun LiteLLM um gegen eine einheitliche API zu modellieren.

In [2]:
from litellm import completion

response = completion(
  model="ollama/qwen3", 
  messages=[{"role": "user", "content": "Why is the sky blue?"}],
)

print(response.choices[0].message.content)

The sky appears blue due to a phenomenon called **Rayleigh scattering**, which involves how sunlight interacts with the Earth's atmosphere. Here's a simplified breakdown:

1. **Sunlight Composition**: Sunlight is a mix of all visible colors (red, orange, yellow, green, blue, indigo, violet), each with different wavelengths. Shorter wavelengths (blue/violet) scatter more easily than longer ones (red/orange).

2. **Scattering in the Atmosphere**: When sunlight enters Earth's atmosphere, gas molecules (like nitrogen and oxygen) scatter the shorter wavelengths more. This scattered blue light travels in all directions, making the sky appear blue to observers on the ground.

3. **Why Blue, Not Violet?**  
   - **Human Eye Sensitivity**: Our eyes are more sensitive to blue light than violet.  
   - **Sunlight Intensity**: The Sun emits more blue light than violet.  
   - **Absorption**: Some violet light is absorbed by the upper atmosphere, reducing its contribution to the sky's color.

4. **

/home/thomi/Lernen/Python/ai-agent-from-scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content="The sky ... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


## Statusmanagement mit APIs

Wir müssen uns den Verlauf der Konversation explizit merken. Ansonsten fangen wir mit jedem API-Call bei 

In [3]:
from litellm import completion

# First call
response1 = completion(
    model="ollama/qwen3",
    messages=[{"role": "user", "content": "My name is Thorsten."}]
)
print(response1.choices[0].message.content)
 
# Second call
response2 = completion(
    model="ollama/qwen3",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print(response2.choices[0].message.content)

Hello, Thorsten! Nice to meet you. How can I assist you today? 😊


/home/thomi/Lernen/Python/ai-agent-from-scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Hello, T... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Hello! I don't have your name yet. Could you please tell me your name so I can address you properly? 😊


/home/thomi/Lernen/Python/ai-agent-from-scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content="Hello! I... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Nun merken wir uns explizit den Verlauf unserer Konversation, und teilen sie dem LLM mit:

In [4]:
from litellm import completion

messages = []

# First call
messages.append({"role": "user", "content": "My name is Thorsten."})

response1 = completion(model="ollama/qwen3", messages=messages)
assistant_message1 = response1.choices[0].message.content;
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)
 
# Second call
messages.append({"role": "user", "content": "What is my name?"})
response1 = completion(model="ollama/qwen3", messages=messages)
assistant_message1 = response1.choices[0].message.content;
messages.append({"role": "assistant", "content": assistant_message1})
print(assistant_message1)

Hello, Thorsten! How can I assist you today? 😊
Your name is Thorsten! 😊 How can I assist you today?


/home/thomi/Lernen/Python/ai-agent-from-scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='Your nam... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


## Generiere strukturiere Ausgaben mit Pydantic

Damit wir die Ausgaben durch Tools verarbeiten können, möchten wir Antworten gemäß einem definierten JSON-Schema erzeugen.

In [5]:
from pydantic import BaseModel
from litellm import completion
 
class ExtractedInfo(BaseModel):
    name: str
    email: str
    phone: str | None = None
 
response = completion(
    model="ollama/qwen3",
    messages=[{
        "role": "user", 
        "content": "My name is John Smith, my email is john@example.com, and my phone is 555-1234."
    }],
    response_format=ExtractedInfo
)
 
result = response.choices[0].message.content

print(result)

{"name": "John Smith", "email": "john@example.com", "phone": "555-1234"}



/home/thomi/Lernen/Python/ai-agent-from-scratch/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='{"name":... reasoning_content=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...reasoning_content=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


## Make asynchronous LLM calls

In [ ]:
import asyncio
from litellm import acompletion
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')

async def get_response(prompt: str) -> str:
    response = await acompletion(
        model="ollama/qwen3",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

prompts = [
    "What is 2 + 2?",
    "What is the capital of Germany?",
    "Who wrote Romeo and Juliet?"
]

tasks = [get_response(p) for p in prompts]
results = await asyncio.gather(*tasks)

for prompt, answer in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {answer}\n")

Q: What is 2 + 2?
A: The result of 2 + 2 is **4**.

Q: What is the capital of Germany?
A: The capital of Germany is **Berlin**. It has been the capital since the unification of Germany in 1871 and remains the political, cultural, and economic center of the country.

Q: Who wrote Romeo and Juliet?
A: "Romeo and Juliet" was written by **William Shakespeare**, an English playwright and poet from the late 16th and early 17th centuries. The play is one of his most famous works, first published in 1597 as *The Most Excellent and Lamentable Tragedy of Romeo and Juliet*. 

While Shakespeare is universally credited as the author, there have been historical debates (known as the "Shakespeare authorship question") suggesting alternative candidates, such as Edward de Vere, Sir Francis Bacon, or Christopher Marlowe. However, these theories are not widely accepted by mainstream scholars. The overwhelming consensus among literary historians and experts is that Shakespeare wrote the play.



In [ ]:
semaphore = asyncio.Semaphore(10)

async def call_llm(prompt: str) -> str:
    """Call the LLM with automatic retry and a semaphore to limit concurrency."""
    async with semaphore:
        response = await acompletion(
            model="ollama/qwen3",
            messages=[{"role": "user", "content": prompt}],
            num_retries=3
        )
        return response.choices[0].message.content

# Now, only 10 API calls run cuncurrently
prompts = [f"What is {i} + {i}?" for i in range(100)]
tasks = [call_llm(p) for p in prompts]

results  = await asyncio.gather(*tasks, return_exceptions=True)

for prompt, answer in zip(prompts, results):
    print(f"Q: {prompt}")
    print(f"A: {answer}\n")

SyntaxError: unmatched ']' (4203058635.py, line 15)